# Fine-tune Qwen3-8B как учитель для Countdown

```
Цель    : обучить Qwen3-8B решать Countdown (3-4 числа)
Данные  : verified_Qwen2.5-7B-Instruct (качественный CoT)
Метод   : SFT + LoRA 4-bit через Unsloth
GPU     : RTX 3060 Laptop 6GB

После обучения Qwen3-4B будет использоваться для:
  - генерации данных с 5-6 числами (знает задачу → хороший CoT)
  - передачи знаний студенту Gemma-3-1b
```

## Ячейка 1 — Импорты и конфигурация

In [ ]:
import os
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.chdir('/home/maki')

import re
import json
import time
import random
import operator
import numpy as np
from pathlib import Path
from collections import Counter

from huggingface_hub import login
HF_TOKEN = "HF"
login(token=HF_TOKEN)
print("HuggingFace авторизован")

CFG = {
    "n_3_from_7b"  : 4000,   # из verified_Qwen2.5-7B-Instruct
    "n_4_from_7b"  : 4000,
    "n_3_from_4b"  : 4000,    # из verified_Qwen3-4B-Instruct-2507
    "n_4_from_4b"  : 4000,
    
    # Модель
    "model_name"     : "./models/qwen3-4b-bnb-4bit",
    "output_dir"     : "./qwen3-4b-countdown/checkpoints",
    "lora_save_path" : "./qwen3-4b-countdown/lora",
    "gguf_save_path" : "./qwen3-4b-countdown-gguf",

    # Данные
    "dataset_name"   : "HuggingFaceTB/Countdown-Task-GOLD",
    "dataset_config" : "verified_Qwen3-4B-Instruct-2507",
    "val_split"      : 0.05,

    # Токенизатор
    "max_seq_length" : 1024,   # CoT от 7B модели обычно 200-400 токенов

    # LoRA — минимальный rank для экономии памяти
    "lora_r"         : 16,
    "lora_alpha"     : 16,

    # Обучение — агрессивная экономия памяти
    "learning_rate"  : 2e-4,
    "batch_size"     : 2,     # минимум для 6GB
    "grad_accum"     : 8,    # эффективный batch = 16
    "num_epochs"     : 1,
    "warmup_ratio"   : 0.05,
    "lr_scheduler"   : "cosine",
    "optim"          : "adamw_8bit",
    "max_grad_norm"  : 1.0,
    "bf16"           : True,
    "weight_decay"   : 0.01,

    # Сохранение
    "save_steps"     : 40,
    "save_total_limit": 3,
    "logging_steps"  : 10,

    "seed"           : 42,
}

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
Path(CFG["output_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CFG, indent=2))

In [ ]:
!ls

## Ячейка 2 — Вспомогательные функции

In [ ]:
def validate_equation(eq_str, nums, target):
    try:
        allowed = set("0123456789 +-*/().")
        if not set(eq_str) <= allowed:
            return False
        nums_in_eq = [int(x) for x in re.findall(r'\d+', eq_str)]
        nums_avail = sorted(nums)
        for n in sorted(nums_in_eq):
            if n not in nums_avail:
                return False
            nums_avail.remove(n)
        return abs(eval(eq_str) - target) < 1e-6
    except Exception:
        return False


def extract_equation(text):
    m = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL)
    if not m:
        return None
    raw = m.group(1).strip()
    if '=' in raw:
        return raw.split('=')[0].strip()
    return raw


print("Вспомогательные функции загружены")

## Ячейка 3 — Загрузка датасета

In [ ]:
from datasets import load_dataset, Dataset

def load_and_prepare_dataset(cfg):
    print(f"Загружаем {cfg['dataset_name']} [{cfg['dataset_config']}]...")
    ds = load_dataset(
        cfg["dataset_name"],
        cfg["dataset_config"],
        split="train"
    )
    print(f"  Загружено: {len(ds)} примеров")

    # Валидация
    valid = []
    for ex in ds:
        msgs = ex["messages"]
        asst = next((m for m in msgs if m["role"] == "assistant"), None)
        if asst is None:
            continue
        eq = extract_equation(asst["content"])
        if eq and validate_equation(eq, ex["nums"], ex["target"]):
            valid.append({
                "target"  : ex["target"],
                "nums"    : list(ex["nums"]),
                "messages": list(ex["messages"]),
            })

    print(f"  Валидных: {len(valid)} (отфильтровано: {len(ds) - len(valid)})")

    dist = Counter(len(ex["nums"]) for ex in valid)
    print("  Распределение:")
    for k in sorted(dist):
        print(f"    {k} чисел: {dist[k]} ({dist[k]/len(valid)*100:.1f}%)")

    random.shuffle(valid)
    val_size = int(len(valid) * cfg["val_split"])
    ds_all = Dataset.from_list(valid)
    split = ds_all.train_test_split(test_size=val_size, seed=cfg["seed"])

    print(f"  Train: {len(split['train'])} | Val: {len(split['test'])}")
    return split["train"], split["test"]


ds_train, ds_val = load_and_prepare_dataset(CFG)

## Ячейка 4 — Загрузка модели

In [ ]:
import torch
from unsloth import FastModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

# Мониторинг памяти
def print_gpu_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved  = torch.cuda.memory_reserved()  / 1024**3
        print(f"  GPU: allocated={allocated:.2f}GB reserved={reserved:.2f}GB")

print("Загружаем Qwen3-8B...")
print_gpu_memory()

model, tokenizer = FastModel.from_pretrained(
    model_name     = CFG["model_name"],
    max_seq_length = CFG["max_seq_length"],
    load_in_4bit   = True,
    device_map     = "auto", 
    local_files_only = True,
)

print("После загрузки модели:")
print_gpu_memory()

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r                          = CFG["lora_r"],
    lora_alpha                 = CFG["lora_alpha"],
    lora_dropout               = 0,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = CFG["seed"],
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"\nМодель загружена")
print(f"  LoRA trainable: {trainable:,} ({trainable/total*100:.2f}%)")
print(f"  r={CFG['lora_r']}, alpha={CFG['lora_alpha']}")
print("После LoRA:")
print_gpu_memory()

## Ячейка 5 — Форматирование и обучение

In [ ]:
def formatting_func(examples):
    """
    Форматирует примеры через chat template Qwen3.
    Qwen3 использует ChatML: <|im_start|>role\ncontent<|im_end|>
    """
    if isinstance(examples["messages"][0], dict):
        all_messages = [examples["messages"]]
    else:
        all_messages = examples["messages"]

    texts = []
    for msgs in all_messages:
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return texts

In [ ]:
class PrintLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs: return
        step = state.global_step
        loss = logs.get("loss", "")
        lr   = logs.get("learning_rate", "")
        if loss:
            print(f"  step={step:>4} | loss={loss:.4f} | lr={lr:.2e}")


eff_batch   = CFG["batch_size"] * CFG["grad_accum"]
total_steps = len(ds_train) // eff_batch * CFG["num_epochs"]
print(f"Эффективный batch: {eff_batch}")
print(f"Всего шагов: {total_steps}")

training_args = SFTConfig(
    output_dir                  = CFG["output_dir"],
    num_train_epochs            = CFG["num_epochs"],
    per_device_train_batch_size = CFG["batch_size"],
    gradient_accumulation_steps = CFG["grad_accum"],
    learning_rate               = CFG["learning_rate"],
    optim                       = CFG["optim"],
    weight_decay                = CFG["weight_decay"],
    max_grad_norm               = CFG["max_grad_norm"],
    warmup_ratio                = CFG["warmup_ratio"],
    lr_scheduler_type           = CFG["lr_scheduler"],
    bf16                        = CFG["bf16"],
    max_seq_length              = CFG["max_seq_length"],
    packing                     = True,
    eval_strategy               = "no",
    save_strategy               = "steps",
    save_steps                  = CFG["save_steps"],
    save_total_limit            = CFG["save_total_limit"],
    logging_steps               = CFG["logging_steps"],
    report_to                   = "none",
    dataloader_num_workers      = 0,
    dataloader_pin_memory       = False,
    dataset_num_proc            = 0,
    seed                        = CFG["seed"],
)

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = ds_train,
    args             = training_args,
    formatting_func  = formatting_func,
    callbacks        = [PrintLossCallback()],
)

# Маскируем промпт — обучаем только на ответах
# Qwen3 использует ChatML формат
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

print("Trainer готов")

## Ячейка 6 — Запуск обучения

In [ ]:
print("\nНачинаем обучение...\n")
print_gpu_memory()

start  = time.time()
result = trainer.train()
elapsed = time.time() - start

print(f"\nОбучение завершено за {elapsed/60:.1f} минут")
print(f"  Train loss: {result.training_loss:.4f}")
print(f"  Шагов:      {result.global_step}")
print("\nПамять после обучения:")
print_gpu_memory()

In [1]:
import os
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1' # Оставляем, чтобы не ходить в интернет
os.chdir('/home/maki')

from unsloth import FastModel

# Путь к вашей локальной БАЗОВОЙ 4-bit модели
base_model_path = "./models/qwen3-4b-bnb-4bit" 

# Путь к папке с вашим обученным LoRA адаптером
checkpoint_path = "./qwen3-4bit-countdown/checkpoints/checkpoint-200"

print("1. Загружаем базовую 4-bit модель...")
# Загружаем base model так же, как при обучении: load_in_4bit=True
model, tokenizer = FastModel.from_pretrained(
    model_name       = base_model_path,
    max_seq_length   = 1024,
    load_in_4bit     = True,   # Оставляем True, чтобы загрузить квантованную базу
    local_files_only = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
1. Загружаем базовую 4-bit модель...
==((====))==  Unsloth 2026.4.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3060 Laptop GPU. Num GPUs = 1. Max memory: 6.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 8.6. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post1. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

./models/qwen3-4b-bnb-4bit does not have a padding token! Will use pad_token = <<<|PAD_TOKEN|>>>.


In [7]:
print("2. Загружаем и прикрепляем обученный LoRA адаптер...")
# Правильный способ - использовать load_adapter
model.load_adapter(checkpoint_path, adapter_name="qwen_lora", local_files_only=True)

2. Загружаем и прикрепляем обученный LoRA адаптер...


TypeError: LoadStateDictConfig.__init__() got an unexpected keyword argument 'local_files_only'

In [6]:
model.load_adapter?

Signature:
model.load_adapter(
    peft_model_id: str | None = None,
    adapter_name: str | None = None,
    peft_config: dict[str, typing.Any] | None = None,
    adapter_state_dict: dict[str, 'torch.Tensor'] | None = None,
    low_cpu_mem_usage: bool = False,
    is_trainable: bool = False,
    hotswap: Union[bool, Literal['auto']] = 'auto',
    local_files_only: bool = False,
    adapter_kwargs: dict[str, typing.Any] | None = None,
    load_config: Optional[ForwardRef('LoadStateDictConfig')] = None,
    **kwargs,
) -> None
Docstring:
Load adapter weights from file or remote Hub folder. If you are not familiar with adapters and PEFT methods, we
invite you to read more about them on PEFT official documentation: https://huggingface.co/docs/peft

Requires PEFT to be installed as a backend to load the adapter weights.

Args:
    peft_model_id (`str`, *optional*):
        The identifier of the model to look for on the Hub, or a local path to the saved adapter config file
        and adapt

In [ ]:
print("3. Объединяем (merging) LoRA с базовой моделью...")
# Объединяем. После этого шага модель "распаковывается" в FP16
model = model.merge_and_unload()


In [ ]:
print("4. Сохраняем объединенную модель в формате GGUF...")
# Теперь сохраняем в GGUF
model.save_pretrained_gguf(
    "/home/maki/qwen3-countdown-gguf",
    tokenizer,
    quantization_method = "q4_k_m",
)
print("Готово!")

In [1]:
import os
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.chdir('/home/maki')

from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name="./models/qwen3-4b-bnb-4bit",
    adapter_name="./qwen3-4b-countdown/checkpoints/checkpoint-200",
    load_in_4bit=False,
    attn_implementation="eager",
)

model = model.merge_and_unload()

model.save_pretrained("merged_model")
tokenizer.save_pretrained("merged_model")

model.save_pretrained_gguf(
    "merged_model",
    tokenizer,
    quantization_method="q4_k_m",
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3060 Laptop GPU. Num GPUs = 1. Max memory: 6.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 8.6. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post1. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

./models/qwen3-4b-bnb-4bit does not have a padding token! Will use pad_token = <<<|PAD_TOKEN|>>>.


AttributeError: 'Qwen3ForCausalLM' object has no attribute 'merge_and_unload'

In [ ]:
from unsloth import FastModel
from peft import PeftModel

checkpoint_path = os.path.join(CFG["output_dir"], "checkpoint-200")

model, tokenizer = FastModel.from_pretrained(
    model_name          = checkpoint_path,  # загружает базу + LoRA сразу
    max_seq_length      = CFG["max_seq_length"],
    load_in_4bit        = False,
    local_files_only    = True,
    attn_implementation = "eager",
)

# Сохраняем в GGUF
model.save_pretrained_gguf(
    CFG["gguf_save_path"],
    tokenizer,
    quantization_method = "q4_k_m",
)

In [ ]:
from unsloth import FastModel
from peft import PeftModel

checkpoint_path = os.path.join(CFG["output_dir"], "checkpoint-200")

model, tokenizer = FastModel.from_pretrained(
    model_name          = checkpoint_path,  # загружает базу + LoRA сразу
    max_seq_length      = CFG["max_seq_length"],
    load_in_4bit        = True,
    local_files_only    = True
)

## Ячейка 7 — Быстрая проверка качества

In [ ]:
import torch

In [ ]:
FastModel.for_inference(model)
model.eval()

# Тест на нескольких примерах из val
n_check = 20
correct = 0
indices = random.sample(range(len(ds_val)), min(n_check, len(ds_val)))
print(len(indices))

for i, idx in enumerate(indices):
    print(i)
    ex = ds_val[idx]
    msgs = ex["messages"]
    user_part = [m for m in msgs if m["role"] in ("system", "user")]
    prompt = tokenizer.apply_chat_template(
        user_part, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = 1024,
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    eq = extract_equation(generated)
    if eq and validate_equation(eq, ex["nums"], ex["target"]):
        correct += 1

print(f"Val accuracy (n={n_check}): {correct}/{n_check} = {correct/n_check*100:.1f}%")

## Ячейка 8 — Сохранение модели

In [ ]:
# Сохраняем LoRA адаптеры
model.save_pretrained(CFG["lora_save_path"])
tokenizer.save_pretrained(CFG["lora_save_path"])
print(f"LoRA сохранены: {CFG['lora_save_path']}")

# Сохраняем GGUF для использования как учителя через llama-server
print(f"\nСохраняем GGUF (q4_k_m)...")
model.save_pretrained_gguf(
    CFG["gguf_save_path"],
    tokenizer,
    quantization_method = "q4_k_m",
)
print(f"GGUF сохранён: {CFG['gguf_save_path']}")
print("\nГотово! Теперь можно использовать GGUF как учителя для генерации")
print("данных с 5-6 числами через llama-server.")